In [ ]:
import torch
import torch.nn as nn
from transformers import (
    CLIPVisionModel,
    CLIPVisionConfig,
    ViTModel,
    T5EncoderModel,
    T5ForConditionalGeneration,
    CLIPProcessor
)

from transformers.modeling_outputs import BaseModelOutput

from peft import LoraConfig, get_peft_model, TaskType
import os 
from torch.amp import autocast, GradScaler
from tqdm import tqdm

from torch.utils.data import DataLoader
from Modules.retrieval_module import Retriever
from Modules.FusionVLM import FusionVLM
from Modules.datasets import VLMDataset, VLMDataCollator

In [ ]:
TRAIN_IMAGE_DIR = "dataset/flickr30k_images/train"
TRAIN_CAPTIONS_DIR = "dataset/captions-train.csv"

TEST_IMAGE_DIR = "dataset/flickr30k_images/test"
TEST_CAPTIONS_DIR = "dataset/captions-test.csv"

FAISS_PATH = "flickr30k_clip_images.faiss"
TRAIN_METADATA_PATH = "train_metadata.json"
TEST_METADATA_PATH = "test_metadata.json"

CHECKPOINT_DIR = "FusionVLM"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
T5_MODEL_NAME = "t5-base"

In [ ]:
# import json
# import random
# from pathlib import Path
# from PIL import Image
# from torch.utils.data import Dataset
# import torch


# class VLMDataset(Dataset):
#     def __init__(
#         self,
#         image_dir,
#         ref_image_dir,
#         metadata_path,
#         retriever,
#         caption_prompt="A Picture of",
#         num_similar_captions=3,
#         transform=None,
#     ):
#         """
#         image_dir: folder with all images
#         metadata_path: JSON with {idx: {"image_name": str, "captions": List[str], "similar_images": List[int]}}
#         retriever: object with method retrieve_captions(idx) -> List[str]
#         """
#         self.retriever = retriever
#         self.image_base_path = Path(image_dir)
#         self.ref_image_base_path = Path(ref_image_dir)
#         self.caption_prompt = caption_prompt
#         self.num_similar_captions = num_similar_captions
#         self.transform = transform

#         with open(metadata_path, "r", encoding="utf-8") as f:
#             self.metadata = json.load(f)


#     def __len__(self):
#         return len(self.metadata)


#     def _load_image(self, image_name, base_path):
#         path = base_path / image_name
#         image = Image.open(path).convert("RGB")
#         if self.transform:
#             image = self.transform(image)
#         return image


#     def _sample_similar_captions(self, neighbors):
#         if len(neighbors) == 0:
#             return []

#         k = min(self.num_similar_captions, len(neighbors))
#         sampled = random.sample(neighbors, k)
#         captions = []
#         for idx in sampled:
#             caps = self.retriever.retrieve_captions(idx)
#             captions.append(random.choice(caps))
#         return captions


#     def _build_prompt(self, similar_captions):
#         if len(similar_captions) == 0:
#             return self.caption_prompt
#         context = "\n".join(f"- {c}" for c in similar_captions)
#         return f"Similar images are described as:\n{context}\n\n{self.caption_prompt}"


#     def __getitem__(self, idx):
#         img_data = self.metadata[str(idx)]
        
#         # Query image
#         query_image = self._load_image(img_data["image_name"], self.image_base_path)
        
#         # Retrieved image (take first similar image)
#         retrieved_idx = img_data["similar_images"][0]
#         retrieved_image_name = self.retriever.retrieve_image_name(retrieved_idx)
#         retrieved_image = self._load_image(retrieved_image_name, self.ref_image_base_path)
        
#         # Retrieved captions
#         retrieved_captions = self._sample_similar_captions(img_data["similar_images"])
        
#         # Prompt for training
#         prompt = self._build_prompt(retrieved_captions)
        
#         # Target caption (longest one)
#         target_caption = max(img_data["captions"], key=len)
        
        
#         return {
#             "query_image": query_image,
#             "retrieved_image": retrieved_image,
#             # "retrieved_captions": retrieved_captions,
#             "prompt": prompt,
#             "target_caption": target_caption,
#         }


# class VLMDataCollator:
#     def __init__(self, processor, tokenizer, max_seq_len=128, device="cuda"):
#         """
#         processor: a HuggingFace processor with vision & text capabilities
#         """
#         self.processor = processor
#         self.tokenizer = tokenizer
#         self.device = device
#         self.max_seq_len = max_seq_len

#     def __call__(self, batch):
#         query_images = [b["query_image"] for b in batch]
#         retrieved_images = [b["retrieved_image"] for b in batch]
#         prompts = [b["prompt"] for b in batch]
#         targets = [b["target_caption"] for b in batch]

#         # Process images (stack query + retrieved along batch dimension)
#         # Some VLMs expect separate keys for query and retrieved images
#         pixel_values = self.processor(
#             images=query_images,
#             return_tensors="pt"
#         ).pixel_values

#         retrieved_pixel_values = self.processor(
#             images=retrieved_images,
#             return_tensors="pt"
#         ).pixel_values

#         # Process text prompts
#         inputs = self.tokenizer(
#             prompts,
#             padding="longest",
#             padding_side='right',
#             truncation=True,
#             max_length=self.max_seq_len,
#             return_tensors="pt"
#         )

#         labels = self.tokenizer(
#             targets,
#             padding="longest",
#             padding_side='right',
#             truncation=True,
#             max_length=self.max_seq_len,
#             return_tensors="pt"
#         ).input_ids
        

#         # ignore padding tokens in loss
#         labels[labels == self.tokenizer.pad_token_id] = -100

#         return {
#             "query_pixel_values": pixel_values.to(self.device),
#             "retrieved_pixel_values": retrieved_pixel_values.to(self.device),
#             "input_ids": inputs.input_ids.to(self.device),
#             "attention_mask": inputs.attention_mask.to(self.device),
#             "labels": labels.to(self.device),
#         }


In [ ]:
# from transformers import T5TokenizerFast

CLIP_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME, use_fast=True)
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME)
collator = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)

In [ ]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [ ]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [ ]:

# # class FusionBlock(nn.Module):
# #     def __init__(self, hidden_dim, num_heads):
# #         super().__init__()
# #         self.cross_attn = nn.MultiheadAttention(
# #             embed_dim=hidden_dim, num_heads=num_heads, batch_first=True
# #         )
# #         self.ln1 = nn.LayerNorm(hidden_dim)
# #         self.ff = nn.Sequential(
# #             nn.Linear(hidden_dim, hidden_dim * 4),
# #             nn.GELU(),
# #             nn.Linear(hidden_dim * 4, hidden_dim),
# #         )
# #         self.ln2 = nn.LayerNorm(hidden_dim)

# #     def forward(self, text_hidden, vision_hidden):
# #         attn_out, _ = self.cross_attn(
# #             query=text_hidden, key=vision_hidden, value=vision_hidden
# #         )
# #         x = self.ln1(text_hidden + attn_out)
# #         x = self.ln2(x + self.ff(x))
# #         return x
    

# class FusionBlock(nn.Module):
#     def __init__(self, hidden_dim, num_heads):
#         super().__init__()

#         self.cross_attn = nn.MultiheadAttention(
#             hidden_dim, num_heads, batch_first=True
#         )
#         self.self_attn = nn.MultiheadAttention(
#             hidden_dim, num_heads, batch_first=True
#         )

#         self.ln1 = nn.LayerNorm(hidden_dim)
#         self.ln2 = nn.LayerNorm(hidden_dim)
#         self.ln3 = nn.LayerNorm(hidden_dim)

#         self.ff = nn.Sequential(
#             nn.Linear(hidden_dim, hidden_dim * 4),
#             nn.GELU(),
#             nn.Linear(hidden_dim * 4, hidden_dim),
#         )

#     def forward(self, text_hidden, vision_hidden):
#         # cross-attention
#         x, _ = self.cross_attn(
#             query=text_hidden, key=vision_hidden, value=vision_hidden
#         )
#         x = self.ln1(text_hidden + x)

#         # self-attention
#         sa, _ = self.self_attn(x, x, x)
#         x = self.ln2(x + sa)

#         # feed-forward
#         x = self.ln3(x + self.ff(x))
#         return x



# class FusionVLM(nn.Module):
#     def __init__(self, vision_encoder_name: str, text_encoder_name: str, text_decoder_name: str,
#                  num_fusion_blocks: int, num_heads: int = 8, use_local_files=True):
#         super().__init__()

#         # Vision encoder
#         vision_config = CLIPVisionConfig()
#         self.vision_encoder = CLIPVisionModel.from_pretrained(vision_encoder_name, config=vision_config, local_files_only=use_local_files)
#         self.vision_dim = self.vision_encoder.config.hidden_size

#         # Text encoder
#         self.text_encoder = T5EncoderModel.from_pretrained(text_encoder_name, local_files_only=use_local_files)
#         self.text_dim = self.text_encoder.config.d_model

#         # Text decoder (LM head)
#         self.text_decoder = T5ForConditionalGeneration.from_pretrained(text_decoder_name, local_files_only=use_local_files)

#         # Project vision → text space
#         self.vision_proj = nn.Linear(self.vision_dim, self.text_dim)

#         # Fusion blocks
#         self.fusion_blocks = nn.ModuleList(
#             [FusionBlock(self.text_dim, num_heads) for _ in range(num_fusion_blocks)]
#         )

#     def forward(self, query_pixel_values, retrieved_pixel_values, input_ids, attention_mask, labels=None):   
             
#         q_vis = self.vision_encoder(query_pixel_values).last_hidden_state

#         if retrieved_pixel_values is not None:
#             r_vis = self.vision_encoder(retrieved_pixel_values).last_hidden_state
#             vision_hidden = torch.cat([q_vis, r_vis], dim=1)
#         else:
#             vision_hidden = q_vis

#         vision_hidden = self.vision_proj(vision_hidden)


#         # Encode text
#         text_hidden = self.text_encoder(
#             input_ids=input_ids, attention_mask=attention_mask
#         ).last_hidden_state

#         # Fusion
#         for block in self.fusion_blocks:
#             text_hidden = block(text_hidden, vision_hidden)

#         # Decode
#         outputs = self.text_decoder(
#             inputs_embeds=text_hidden,
#             attention_mask=attention_mask,
#             labels=labels,
#             return_dict=True,
#         )

#         return outputs
    
    
#     @torch.no_grad()
#     def generate(self, query_pixel_values, retrieved_pixel_values, input_ids, attention_mask, 
#         max_length=128, num_beams=3, do_sample=False, temperature=1.0, top_p=1.0, **generate_kwargs):
#         """
#         Multimodal text generation using T5 decoder.
#         Fused text+vision embeddings are treated as encoder outputs.
#         """

#         # ---- Vision ----
#         q_vis = self.vision_encoder(query_pixel_values).last_hidden_state

#         if retrieved_pixel_values is not None:
#             r_vis = self.vision_encoder(retrieved_pixel_values).last_hidden_state
#             vision_hidden = torch.cat([q_vis, r_vis], dim=1)
#         else:
#             vision_hidden = q_vis

#         vision_hidden = self.vision_proj(vision_hidden)

#         # ---- Text encoder ----
#         text_hidden = self.text_encoder(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         ).last_hidden_state

#         # ---- Fusion ----
#         for block in self.fusion_blocks:
#             text_hidden = block(text_hidden, vision_hidden)

#         # ---- Pretend fused embeddings are encoder outputs ----
#         encoder_outputs = BaseModelOutput(
#             last_hidden_state=text_hidden
#         )

#         # ---- Generate ----
#         generated_ids = self.text_decoder.generate(
#             encoder_outputs=encoder_outputs,
#             attention_mask=attention_mask,
#             max_length=max_length,
#             num_beams=num_beams,
#             do_sample=do_sample,
#             temperature=temperature,
#             top_p=top_p,
#             **generate_kwargs,
#         )

#         return generated_ids


In [ ]:
# openai/clip-vit-base-patch16
# google/vit-base-patch16-224

# Instantiate and count parameters
model = FusionVLM(
    # vision_encoder_name="openai/clip-vit-base-patch16",
    vision_encoder_name="openai/clip-vit-base-patch32",
    text_encoder_name="t5-base",
    text_decoder_name="t5-base",
    num_fusion_blocks=4,
    use_local_files=True
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")

In [ ]:
def freeze_module(module: torch.nn.Module):
    for p in module.parameters():
        p.requires_grad = False

# Freeze vision encoder
freeze_module(model.vision_encoder)

# Freeze text encoder
freeze_module(model.text_encoder)

# Freeze text decoder
# freeze_module(model.text_decoder)

In [ ]:

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        # Decoder self-attention (language modeling)
        "SelfAttention.q",
        "SelfAttention.v",

        # Decoder cross-attention (fusion output → text)
        "EncDecAttention.q",
        "EncDecAttention.v",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model.text_decoder = get_peft_model(model.text_decoder, lora_config)
model.text_decoder.print_trainable_parameters()


In [ ]:
# Unfreeze the lm head for token generation
for param in model.text_decoder.lm_head.parameters():
    print(param.shape[0]*param.shape[1])
    param.requires_grad = True

In [ ]:
def print_model_param_stats(model: nn.Module):
    total_params = 0
    trainable_params = 0
    frozen_params = 0

    print(f"{'Module':40} {'Total':>12} {'Trainable':>12} {'Frozen':>12}")
    print("-" * 80)

    # Iterate over top-level modules
    for name, module in model.named_children():
        module_total = sum(p.numel() for p in module.parameters())
        module_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
        module_frozen = module_total - module_trainable

        total_params += module_total
        trainable_params += module_trainable
        frozen_params += module_frozen

        print(f"{name:40} {module_total:12,} {module_trainable:12,} {module_frozen:12,}")

    print("-" * 80)
    print(f"{'TOTAL':40} {total_params:12,} {trainable_params:12,} {frozen_params:12,}")


print_model_param_stats(model)

In [ ]:
num_epochs = 1
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

scaler = GradScaler()


In [ ]:


os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model.train()
loss_history = []

for epoch in range(num_epochs):
    batch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)

    for batch in progress_bar:
        optimizer.zero_grad()

        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(
                query_pixel_values=batch["query_pixel_values"],
                retrieved_pixel_values=batch["retrieved_pixel_values"],
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"]
            )
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    epoch_loss = batch_loss / len(train_loader)
    loss_history.append(epoch_loss)
    
    progress_bar.set_postfix(final_loss=epoch_loss)
    print(f"Epoch {epoch+1} Avg Loss: {epoch_loss:.4f}")

    # ---- Backup every 5 epochs ----
    if (epoch + 1) % 5 == 0:
        backup_path = os.path.join(CHECKPOINT_DIR, f"FusionVLM_epoch{epoch+1}.pt")
        torch.save(model.state_dict(), backup_path)


In [ ]:
model.eval()

for batch in train_loader:
    with torch.no_grad():
        outputs = model(
                        query_pixel_values=batch["query_pixel_values"],
                        retrieved_pixel_values=batch["retrieved_pixel_values"],
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        # labels=batch["labels"]
                        labels=None
                    )
    break

logits = outputs.logits

pred_ids = torch.argmax(logits, dim=-1)

texts = T5_tokenizer.batch_decode(
    pred_ids,
    skip_special_tokens=True
)

print(texts)

In [ ]:
outputs.logits.shape

In [ ]:
model.eval()

for batch in test_loader:
    with torch.no_grad():
        # generated_ids = model.generate(
        #     query_pixel_values=batch["query_pixel_values"][0].unsqueeze(0),
        #     retrieved_pixel_values=batch["retrieved_pixel_values"][0].unsqueeze(0),
        #     input_ids=batch["input_ids"][0].unsqueeze(0),             # e.g. "describe the image:"
        #     attention_mask=batch["attention_mask"][0].unsqueeze(0),
        #     max_length=64,
        #     num_beams=4
        # )

        generated_ids = model.generate(
            query_pixel_values=batch["query_pixel_values"],
            retrieved_pixel_values=batch["retrieved_pixel_values"],
            input_ids=batch["input_ids"],             # e.g. "describe the image:"
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=4
        )
        
        captions = T5_tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        print(captions)
        
    break


In [ ]:
generated_ids